# Homework 2 — Fit, Measure, Descend
**CE/ARCE 2XX · AI Literacy for Civil and Architectural Engineering** · **Due:** Sunday 11:59 pm (Arizona time) at the end of Week 5, on Brightspace

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/homework/hw2-fit-measure-descend/HW2_Fit_Measure_Descend.ipynb)

You will do the full Weeks 3–4 recipe yourself: compute a cost and a gradient-descent step **by hand** (Part A), train a one-feature linear regression **from scratch** on building heating loads (Part B), fit a **multiple regression** with scikit-learn and test it properly (Part C), and write a short **engineering memo** with honest caveats (Part D). Read the full handout (README) on GitHub for the rubric. Total: **100 points**; plan about **5 hours**. The AI-use acknowledgment at the end is required: a missing acknowledgment costs 5 points.

### What you will be able to do
1. Compute predictions, errors, the cost J(w, b), the gradients, and one gradient-descent step by hand.
2. Implement `predict`, `compute_cost`, `compute_gradient`, and `gradient_descent`, and choose a learning rate from learning curves.
3. Fit and evaluate a multiple linear regression with a train/test split and MAE, RMSE, and R².
4. Recognize a confounding variable and explain why coefficients are not design rules.
5. Communicate a model's accuracy and limits in a short engineering memo.

### Before you start
1. Click **Open in Colab** above (or open this file from the course GitHub page).
2. In Colab, choose **File → Save a copy in Drive**. Work in *your copy* so nothing is lost.
3. Run cells from top to bottom with **Shift + Enter**. Cells marked `# TODO` are yours to complete.

**Runtime:** the default CPU runtime is fine.
**Data:** `building_energy_efficiency.csv` — 768 simulated building shapes (Tsanas & Xifara 2012, UCI, CC BY 4.0). See the course `data/` folder README.

**AI-use label:** **Mixed — see each part.** Part A is **AI-Free**: no AI tools (or calculators that do the algebra for you) to produce your answers; a basic calculator is fine. Parts B–D are **AI-Assisted**: you may ask an AI tool for explanations or debugging help, and you must say so in the acknowledgment at the end. You are responsible for every line.

In [ ]:
# Setup: run this cell first (Shift + Enter). It loads libraries and a helper to fetch course data.
import os, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BASE_URL = "https://raw.githubusercontent.com/TensorBuildLab/AI-Literacy-CE-ARCE/main/data/"

def get_data(name):
    """Download a course data file once and return its local path, e.g. get_data("concrete_strength.csv")."""
    if not os.path.exists(name):
        folder = os.path.dirname(name)
        if folder:
            os.makedirs(folder, exist_ok=True)
        urllib.request.urlretrieve(BASE_URL + name, name)
    return name

print("Setup complete.")

## Part A — Cost and one descent step by hand (20 points · AI-Free)
Three concrete mixes were tested at 28 days:

| Mix | w/c ratio, x | Strength, y (MPa) |
|---|---|---|
| A | 0.40 | 49 |
| B | 0.50 | 41 |
| C | 0.60 | 36 |

Two candidate lines: **Line 1:** w = −50 MPa, b = 68 MPa. **Line 2:** w = −60 MPa, b = 73 MPa.

Use the course formulas (m = 3):

$$f_{w,b}(x) = w\,x + b \qquad J(w,b) = \frac{1}{2m}\sum_{i=1}^{m}\left(f_{w,b}(x^{(i)}) - y^{(i)}\right)^2$$

$$\frac{\partial J}{\partial w} = \frac{1}{m}\sum_{i=1}^{m}\left(f_{w,b}(x^{(i)}) - y^{(i)}\right)x^{(i)} \qquad
\frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=1}^{m}\left(f_{w,b}(x^{(i)}) - y^{(i)}\right)$$

Show your work in the answer cells (double-click to edit). A table like the one in Lab 03 is the clearest way.

**A1 (6 pts).** For Line 1, compute each prediction, error, and squared error, and then J (with units).

| Mix | x | y (MPa) | f(x) (MPa) | error f(x) − y | error² |
|---|---|---|---|---|---|
| A | 0.40 | 49 | | | |
| B | 0.50 | 41 | | | |
| C | 0.60 | 36 | | | |

J(−50, 68) = 

**A2 (4 pts).** Compute J for Line 2. Which line fits these three mixes better, and how do you know?

*Your answer:* 

**A3 (6 pts).** At Line 1, compute ∂J/∂w and ∂J/∂b. Show the sums.

*Your answer:* 

**A4 (4 pts).** Take **one** gradient-descent step from Line 1 with α = 0.5 (simultaneous update). Give the new w and b, compute the new J, and say in one sentence why both w and b went *down*.

*Your answer:* 

**Verify (optional, no points).** After finishing Part A *by hand*, type your final numbers below and run the check. It only says ✓ or ✗ — it does not reveal the answers.

In [ ]:
J1 = None      # cost of Line 1 (MPa²)
J2 = None      # cost of Line 2 (MPa²)
dj_dw = None   # ∂J/∂w at Line 1
dj_db = None   # ∂J/∂b at Line 1
w_new = None   # after one step with alpha = 0.5 (MPa)
b_new = None   # (MPa)
J_new = None   # cost after the step (MPa²)

In [ ]:
xa, ya = np.array([0.40, 0.50, 0.60]), np.array([49.0, 41.0, 36.0])
_e1 = (-50 * xa + 68) - ya
_wn, _bn = -50 - 0.5 * np.mean(_e1 * xa), 68 - 0.5 * np.mean(_e1)
_truth = {"J1": np.sum(_e1**2) / 6, "J2": np.sum(((-60 * xa + 73) - ya)**2) / 6,
          "dj_dw": np.mean(_e1 * xa), "dj_db": np.mean(_e1), "w_new": _wn, "b_new": _bn,
          "J_new": np.sum(((_wn * xa + _bn) - ya)**2) / 6}
_mine = {"J1": J1, "J2": J2, "dj_dw": dj_dw, "dj_db": dj_db, "w_new": w_new, "b_new": b_new, "J_new": J_new}
for k in _truth:
    ok = _mine[k] is not None and abs(_mine[k] - _truth[k]) < 0.01
    print(f"{k:6s} {'✓ matches' if ok else '✗ recheck (or not entered yet)'}")

## Part B — One-feature linear regression from scratch (35 points · AI-Assisted)
The dataset holds 768 simulated buildings with the same materials; they differ in shape, height, glazing, and
orientation. **Goal:** predict the yearly **heating load** (kWh/m²) from **relative compactness** (RC, no units:
volume-to-surface ratio compared with the most compact shape — higher means more compact).

**B1 (4 pts).** Load the data into `bldg`. Make numpy arrays `x` (relative compactness) and `y` (heating load).
Make a scatter plot of y vs x with a title and axis labels with units.

> 💡 **Hint:** `pd.read_csv(get_data("building_energy_efficiency.csv"))`; `.to_numpy()` turns a column into an array (Lab 03).

In [ ]:
# TODO (B1)
bldg = None
x = None
y = None

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert x is not None and y is not None, "x or y is still None. Finish B1 first."
assert len(x) == 768 and abs(x.mean() - 0.764) < 0.001, "x should be the relative_compactness column as an array (mean about 0.764). Use .to_numpy()."
assert abs(y.mean() - 22.307) < 0.001, "y should be the heating_load_kwh_m2 column as an array (mean about 22.31 kWh/m²)."
print("Correct!")

**B2 (10 pts).** Write the four functions. You may start from your own Lab 03 and Lab 04 code. Keep these
exact names and arguments — the checks use them.

> 💡 **Hint:** Copy from Labs 03–04. The loop in `gradient_descent` has three steps: get both slopes, update w and b together, append the new cost.

In [ ]:
# TODO (B2)
def predict(x, w, b):
    """Prediction f(x) = w*x + b."""
    return None

def compute_cost(x, y, w, b):
    """Cost J(w, b) = (1/2m) * sum of squared errors."""
    return None

def compute_gradient(x, y, w, b):
    """Return (dj_dw, dj_db)."""
    return None, None

def gradient_descent(x, y, w0, b0, alpha, num_iters):
    """Return final w, b, and the list of costs (one before the first step, then one after each step)."""
    w, b = w0, b0
    J_history = [compute_cost(x, y, w, b)]
    # your loop here (no need to record the path this time)
    return w, b, J_history   # 3 values — Lab 04's version also returned the path

In [ ]:
# ✅ Self-check: run this cell to test your work above
xa, ya = np.array([0.40, 0.50, 0.60]), np.array([49.0, 41.0, 36.0])   # Part A data
assert predict(0.5, -50, 68) is not None, "predict returned None. Finish predict first (return w * x + b)."
Jc = compute_cost(xa, ya, -50, 68)
assert Jc is not None, "compute_cost returned None. Finish compute_cost."
assert abs(Jc - 1.5) < 1e-9, f"compute_cost should give 1.5 for Part A's Line 1, but gave {Jc}. Divide by (2 * m), with the parentheses."
gw, gb = compute_gradient(xa, ya, -50, 68)
assert gw is not None and gb is not None, "compute_gradient returned None. Finish compute_gradient."
assert abs(gw - 0.6) < 1e-9 and abs(gb - 1.0) < 1e-9,     f"compute_gradient should give (0.6, 1.0) for Part A's Line 1, but gave ({gw}, {gb}). Errors = predictions - y; dj_dw averages errors * x."
w1, b1, Jh = gradient_descent(xa, ya, -50, 68, 0.5, 1)
assert len(Jh) == 2, f"After 1 step, J_history should hold 2 costs, but it holds {len(Jh)}. Append the new cost inside the loop."
assert abs(w1 - (-50.3)) < 1e-9 and abs(b1 - 67.5) < 1e-9,     f"One step with alpha = 0.5 should give w = -50.3, b = 67.5 (Part A4), but gave {w1}, {b1}. Check the minus signs in the update."
print("All four functions are correct — and they agree with Part A.")

**B3 (8 pts).** Run gradient descent from w = 0, b = 0 with α = 1.0 for 3,000 iterations. Compare with
`np.polyfit`. Plot the fitted line over the data, and compute **MAE**, **RMSE**, and **R²** of the fitted line on all
768 buildings (formulas in Lab 03/04).

> 💡 **Hint:** `gradient_descent(x, y, 0.0, 0.0, 1.0, 3000)`; `np.polyfit(x, y, 1)`; `f = predict(x, w_gd, b_gd)`; `np.mean(np.abs(f - y))`; `np.sqrt(np.mean((f - y)**2))`; R² = `1 - np.sum((f - y)**2) / np.sum((y - np.mean(y))**2)` — mind the parentheses.

In [ ]:
# TODO (B3): replace each None, and write the plt.plot line
w_gd, b_gd, J_hist = None, None, None   # gradient descent from (0, 0), alpha = 1.0, 3000 iterations
w_fit, b_fit = None, None               # np.polyfit on the same x and y
print(f"Gradient descent: w = {w_gd}, b = {b_gd}")
print(f"np.polyfit:       w = {w_fit}, b = {b_fit}")

f = None      # predictions of the gradient-descent line for all 768 buildings
mae = None    # average of |f - y|                         (kWh/m²)
rmse = None   # square root of the average of (f - y)**2  (kWh/m²)
r2 = None     # 1 - sum((f - y)**2) / sum((y - mean(y))**2)
print(f"MAE = {mae}, RMSE = {rmse}, R² = {r2}")

x_line = np.linspace(0.6, 1.0, 50)
plt.figure(figsize=(7, 4))
plt.scatter(x, y, alpha=0.4, s=15, label="buildings")
# plt.plot(...)   <- the line predict(x_line, w_gd, b_gd), in red, with label="gradient-descent line"
plt.xlabel("Relative compactness (–)")
plt.ylabel("Heating load (kWh/m²)")
plt.title("One-feature linear model for heating load")
plt.legend()
plt.show()

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert w_gd is not None and r2 is not None, "w_gd or r2 is still None. Finish B3 first."
assert abs(w_gd - 59.36) < 0.05 and abs(b_gd - (-23.05)) < 0.05,     f"Expected w ≈ 59.36 and b ≈ -23.05 after 3000 steps, got {w_gd:.2f} and {b_gd:.2f}. Use alpha = 1.0, 3000 iterations, start at (0, 0)."
assert abs(r2 - 0.387) < 0.005, f"R² should be about 0.39, but yours is {r2:.3f}. Check the parentheses in 1 - sum(...) / sum(...)."
print("Correct!")

**✍️ Question:** Interpret w in words with units: "For each 0.1 increase in relative compactness, the predicted heating load changes by ___." What are R² and MAE, and is this a strong model? *(3 pts)*

*Your answer:* 

**B4 (8 pts).** Learning-rate experiment. Run gradient descent from (0, 0) for 3,000 iterations with
α = 0.01, 0.1, and 1.0. Plot the three learning curves on one figure (log-scale y-axis, legend, labels) and print
the final J for each. Then run α = 1.3 for 50 iterations and print the first few costs.

> 💡 **Hint:** Inside the loop: `w_a, b_a, J_a = gradient_descent(x, y, 0.0, 0.0, alpha, 3000)` and `plt.plot(J_a, label=f"α = {alpha}")` (indented 4 spaces). After the loop: `gradient_descent(x, y, 0.0, 0.0, 1.3, 50)`.

In [ ]:
# TODO (B4): replace each None, and write the plt.plot line inside the loop
plt.figure(figsize=(8, 4.5))
for alpha in [0.01, 0.1, 1.0]:
    w_a, b_a, J_a = None, None, [None]   # gradient descent from (0, 0) with this alpha, 3000 iterations
    print(f"α = {alpha}: final J = {J_a[-1]}")
    # plt.plot(...)   <- plot J_a with label=f"α = {alpha}"
plt.yscale("log")
plt.xlabel("Iteration")
plt.ylabel("Cost J ((kWh/m²)², log scale)")
plt.title("Learning curves for heating load vs relative compactness")
plt.legend()
plt.show()

w_big, b_big, J_big = None, None, [None]   # gradient descent with alpha = 1.3 for 50 iterations
print("α = 1.3, first costs:", J_big[:6])

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert J_a[-1] is not None and J_big[-1] is not None, "Finish B4 first (J_a or J_big still holds None)."
assert abs(J_a[-1] - 31.154) < 0.01,     f"With alpha = 1.0 and 3000 iterations, the final cost should be about 31.15, but it is {J_a[-1]:.3f}. Check gradient_descent(x, y, 0.0, 0.0, alpha, 3000)."
assert J_big[-1] > J_big[0], "With alpha = 1.3 the cost should GROW from step to step. Check that you used 1.3 and 50 iterations."
print("Correct!")

**✍️ Question:** Which α would you use, and why? Describe what happened with α = 0.01 and with α = 1.3. Compared with the concrete data in Lab 04, why do even good learning rates need hundreds of steps here? (Hint: look at how narrow the range of x is.) *(3 pts)*

*Your answer:* 

**B5 (5 pts).** A hidden variable. Every building is either 3.5 m or 7 m tall. Fit a separate line
(`np.polyfit`) for each height group and print the two slopes. The plotting cell below then draws everything.

> 💡 **Hint:** Filter: `short = bldg[bldg["overall_height_m"] == 3.5]`, same for 7. Then `slope_short, int_short = np.polyfit(short["relative_compactness"], short["heating_load_kwh_m2"], 1)`. `np.polyfit` returns **two** numbers, so unpack them into two names.

In [ ]:
# TODO (B5): slopes for each height group — replace each None
short = None                          # only the 3.5 m buildings
tall = None                           # only the 7 m buildings
slope_short, int_short = None, None   # np.polyfit gives TWO numbers: slope and intercept
slope_tall, int_tall = None, None
print("3.5 m buildings slope:", slope_short, "   7 m buildings slope:", slope_tall)

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert slope_short is not None and slope_tall is not None, "A slope is still None. Finish B5 first."
assert np.ndim(slope_short) == 0 and np.ndim(slope_tall) == 0,     "A slope holds two numbers. np.polyfit returns [slope, intercept]; unpack it: slope_short, int_short = np.polyfit(...)."
assert abs(slope_short - (-30.9)) < 0.1,     f"The 3.5 m slope should be about -30.9, but yours is {slope_short:.1f}. Filter on overall_height_m == 3.5; x (relative_compactness) comes first, y (heating load) second."
assert abs(slope_tall - (-35.7)) < 0.1, f"The 7 m slope should be about -35.7, but yours is {slope_tall:.1f}. Filter on overall_height_m == 7."
print("Correct!")

In [ ]:
plt.figure(figsize=(7, 4.5))
for h, marker in [(3.5, "o"), (7, "^")]:
    grp = bldg[bldg["overall_height_m"] == h]
    s, c = np.polyfit(grp["relative_compactness"], grp["heating_load_kwh_m2"], 1)
    xs = np.linspace(grp["relative_compactness"].min(), grp["relative_compactness"].max(), 20)
    plt.scatter(grp["relative_compactness"], grp["heating_load_kwh_m2"], alpha=0.4, s=15, marker=marker,
                label=f"{h} m tall")
    plt.plot(xs, s * xs + c, linewidth=2, label=f"line within {h} m group (slope {s:.0f})")
xs = np.linspace(0.6, 1.0, 20)
plt.plot(xs, predict(xs, w_gd, b_gd), "k--", label=f"line through all buildings (slope {w_gd:.0f})")
plt.xlabel("Relative compactness (–)")
plt.ylabel("Heating load (kWh/m²)")
plt.title("The overall trend reverses inside each height group")
plt.legend(fontsize=8)
plt.show()

**✍️ Question:** Explain the plot. Why is the overall slope positive while the slope inside each height group is negative? Which direction matches the physics of heat loss through the building envelope? *(5 pts)*

*Your answer:* 

## Part C — Multiple regression with scikit-learn (30 points · AI-Assisted)
Use all 8 design columns to predict heating load. Do **not** use `cooling_load_kwh_m2` (it is another simulation
output — see HW1).

**C1 (6 pts).** Make `X` (8 design columns) and `y_all` (heating load). Split with `train_test_split(test_size=0.2,
random_state=42)`, then standardize with a `StandardScaler` fitted on the training set only.

> 💡 **Hint:** Follow Lab 04, Part 6, step by step.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

design_cols = ["relative_compactness", "surface_area_m2", "wall_area_m2", "roof_area_m2",
               "overall_height_m", "orientation", "glazing_area_ratio", "glazing_distribution"]
# TODO (C1)
X = None
y_all = None
X_train, X_test, y_train, y_test = None, None, None, None
scaler = None
X_train_s = None
X_test_s = None

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert X_train_s is not None and X_test_s is not None, "X_train_s or X_test_s is still None. Finish C1 first."
assert X_train_s.shape == (614, 8) and X_test_s.shape == (154, 8),     f"Expected 614 × 8 (train) and 154 × 8 (test), got {X_train_s.shape} and {X_test_s.shape}. Use test_size=0.2, random_state=42 and the 8 design columns."
assert abs(X_train_s.mean()) < 1e-9, "The scaled training data should have mean 0. Use scaler.fit_transform(X_train), then scaler.transform(X_test)."
print("Correct!")

**C2 (8 pts).** Fit a `LinearRegression` on the scaled training data and predict the training and test
sets (`y_pred_train`, `y_pred_test`). The provided cell then builds a comparison table with a baseline (always predict
the training mean) and the one-feature RC line (fitted on the training set only).

> 💡 **Hint:** Create, `.fit(X_train_s, y_train)`, then `.predict(...)` twice.

In [ ]:
# TODO (C2)
model = None
y_pred_train = None
y_pred_test = None

In [ ]:
# ✅ Self-check: run this cell to test your work above
assert y_pred_train is not None and y_pred_test is not None, "A prediction is still None. Finish C2 first."
assert len(y_pred_train) == 614 and len(y_pred_test) == 154,     f"Expected 614 training and 154 test predictions, got {len(y_pred_train)} and {len(y_pred_test)}. Predict X_train_s and X_test_s."
print("Correct!")

In [ ]:
def metrics(y_true, y_pred):
    """Return MAE, RMSE (kWh/m²), and R²."""
    return (mean_absolute_error(y_true, y_pred), np.sqrt(mean_squared_error(y_true, y_pred)), r2_score(y_true, y_pred))

w_rc, b_rc = np.polyfit(X_train["relative_compactness"], y_train, 1)
rc_pred_test = predict(X_test["relative_compactness"].to_numpy(), w_rc, b_rc)
rows = {
    "Baseline (training mean), test": metrics(y_test, np.full(len(y_test), y_train.mean())),
    "RC line, test": metrics(y_test, rc_pred_test),
    "8 inputs, train": metrics(y_train, y_pred_train),
    "8 inputs, test": metrics(y_test, y_pred_test),
}
pd.DataFrame(rows, index=["MAE (kWh/m²)", "RMSE (kWh/m²)", "R²"]).T.round(3)

**C3 (6 pts).** Plot predicted vs actual heating load for the **test set**, with the dashed "perfect prediction" line, a legend, a title, and axis labels with units.

> 💡 **Hint:** Same as the last plot of Lab 04, with new labels and a 0–45 kWh/m² line.

In [ ]:
# TODO (C3)

**C4 (10 pts).** Run the cell below (coefficients and a check on the geometry columns), then answer the three questions.

In [ ]:
coefs = pd.Series(model.coef_, index=design_cols).round(2)
print("Coefficients (kWh/m² per +1 std of each input):")
print(coefs.to_string())
print(f"Intercept: {model.intercept_:.2f} kWh/m²")
print()
same = np.allclose(bldg["surface_area_m2"], bldg["wall_area_m2"] + 2 * bldg["roof_area_m2"])
print("Is surface area = wall area + 2 × roof area for every building?", same)

**✍️ Question:** (a) Compare the 8-input model with the baseline and the RC line on the test set (use MAE and R²). Is there a sign of overfitting? *(3 pts)*

*Your answer:* 

**✍️ Question:** (b) In Part B the RC slope was **positive**; here the RC coefficient is **negative**. Explain why the sign changed. *(4 pts)*

*Your answer:* 

**✍️ Question:** (c) Surface area equals wall area + 2 × roof area for every building (see the note on redundant inputs at the end of Lab 04, Part 6.4). Which statement is correct? Choose **one** letter and explain your choice in one sentence.<br>**A.** The surface-area coefficient (≈ −3.6) means one extra standard deviation of surface area lowers the heating load by 3.6 kWh/m², whatever the walls and roof do.<br>**B.** The three columns are redundant: many different combinations of their coefficients give exactly the same predictions, so their individual values cannot be interpreted — but the predictions are still fine.<br>**C.** The model is broken, and its predictions cannot be used.<br>**D.** Adding the three coefficients together gives the effect of 1 m² of envelope area. *(3 pts)*

*Your answer:* 

## Part D — Engineering memo (15 points · AI-Assisted)
Write **one paragraph (120–180 words)** to the lead of a design team who asks: *"Can we use a quick
model instead of a full energy simulation to estimate heating load at the concept stage?"* Include:
- what you modeled and with what data (in one sentence);
- your best model's **test** accuracy in kWh/m² (MAE and R²) and how it compares with the one-feature model;
- **at least three specific caveats** (think about where the data came from, what range of designs it covers, the
  confounding you found, and what the coefficients can and cannot tell you);
- a clear recommendation (use it for what, and not for what).

Then complete the AI-use acknowledgment at the end of the notebook (required; a missing acknowledgment costs
5 points).

*Your memo:* 

---
## 🤖 AI-use acknowledgment
Did you use an AI tool (ChatGPT, Gemini, Copilot, Claude, Colab's AI features, etc.) while working on this notebook? (The small model that this notebook's own code runs does not count.)
- If **no**, write: *"I did not use AI tools for this assignment."*
- If **yes**, write: *"I acknowledge the use of [tool and version] to [specific use]. Prompts I used included [paste or summarize]. I checked the output by [how I verified it]. I am responsible for this work."*

Same template as the course [AI-use guide](https://github.com/TensorBuildLab/AI-Literacy-CE-ARCE/blob/main/resources/ai-use-guide.md).

*Your statement:* 

## 📤 How to submit
1. **Runtime → Restart session and run all** — make sure everything runs without errors (cells that load a model take a few extra minutes).
2. **File → Download → Download .ipynb**.
3. Upload the `.ipynb` file to the matching assignment on **Brightspace**.